# Exoplanet Data Analysis for Demographic Study

**Author:** Aradhya Haldikar  
**Date:** March 2026  
**Data source:** NASA Exoplanet Archive (DOI: 10.26133/NEA13)

This notebook downloads confirmed exoplanet data from the NASA Exoplanet Archive, computes quality flags based on measurement uncertainties, and saves a filtered dataset for further analysis. The output file `exoplanet_data_with_quality.csv` is used in the accompanying paper *"A Self‑Directed Census: Analyzing 6,105 Confirmed Exoplanets from First Principles"*.

## 1. Install required packages
We need `astroquery` to fetch data from the NASA Exoplanet Archive.

In [1]:
!pip install astroquery
import pandas as pd
import numpy as np
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

## 2. Fetch confirmed planets data
We query the `ps` table (Planetary Systems) with `default_flag=1` to get only confirmed planets with their default parameter sets. The columns we need are:
- `pl_name`: planet name
- `pl_masse`: planet mass [Earth masses]
- `pl_masseerr1`, `pl_masseerr2`: upper and lower mass uncertainties
- `pl_rade`: planet radius [Earth radii]
- `pl_radeerr1`, `pl_radeerr2`: upper and lower radius uncertainties

In [2]:
print("Fetching confirmed planets data...")
df = NasaExoplanetArchive.query_criteria(
    table="ps",
    select="pl_name, pl_masse, pl_masseerr1, pl_masseerr2, pl_rade, pl_radeerr1, pl_radeerr2",
    where="default_flag=1",
    cache=False
).to_pandas()

print(f"Total confirmed planets: {len(df)}")
print(f"Planets with mass data (non‑null): {df['pl_masse'].notna().sum()}")
print(f"Planets with radius data (non‑null): {df['pl_rade'].notna().sum()}\n")

Fetching confirmed planets data...
Total confirmed planets: 6128
Planets with mass data (non‑null): 2051
Planets with radius data (non‑null): 4561



## 3. Compute quality flags based on uncertainty

We define a planet as having **reliable mass** if:
1. Both mass errors are present (not null)
2. The fractional uncertainty (larger error divided by the value) is less than 30%

Same for radius. Planets without errors do **not** pass the quality cut.

In [3]:
# Mass quality
mass_ok = df['pl_masse'].notna() & df['pl_masseerr1'].notna() & df['pl_masseerr2'].notna()
df_mass = df[mass_ok].copy()
if len(df_mass) > 0:
    df_mass['mass_frac_err'] = np.maximum(df_mass['pl_masseerr1'], df_mass['pl_masseerr2']) / df_mass['pl_masse']
    mass_good = df_mass[df_mass['mass_frac_err'] < 0.3].shape[0]
else:
    mass_good = 0
print(f"Planets with both mass errors present: {len(df_mass)}")
print(f"Planets with mass uncertainty <30%: {mass_good}")

# Radius quality
rade_ok = df['pl_rade'].notna() & df['pl_radeerr1'].notna() & df['pl_radeerr2'].notna()
df_rade = df[rade_ok].copy()
if len(df_rade) > 0:
    df_rade['rade_frac_err'] = np.maximum(df_rade['pl_radeerr1'], df_rade['pl_radeerr2']) / df_rade['pl_rade']
    rade_good = df_rade[df_rade['rade_frac_err'] < 0.3].shape[0]
else:
    rade_good = 0
print(f"\nPlanets with both radius errors present: {len(df_rade)}")
print(f"Planets with radius uncertainty <30%: {rade_good}")

Planets with both mass errors present: 1891
Planets with mass uncertainty <30%: 1413

Planets with both radius errors present: 4183
Planets with radius uncertainty <30%: 3708


## 4. Add quality flags to the full dataset

We create two new columns:
- `mass_quality`: True if mass uncertainty <30%, False otherwise
- `rade_quality`: True if radius uncertainty <30%, False otherwise

Planets without errors get `False` (they do not pass the quality cut).

In [4]:
# Compute fractional errors for all rows (NaN where errors missing)
mass_ok = df['pl_masse'].notna() & df['pl_masseerr1'].notna() & df['pl_masseerr2'].notna()
rade_ok = df['pl_rade'].notna() & df['pl_radeerr1'].notna() & df['pl_radeerr2'].notna()

df['mass_frac_err'] = np.nan
df.loc[mass_ok, 'mass_frac_err'] = np.maximum(df.loc[mass_ok, 'pl_masseerr1'], df.loc[mass_ok, 'pl_masseerr2']) / df.loc[mass_ok, 'pl_masse']

df['rade_frac_err'] = np.nan
df.loc[rade_ok, 'rade_frac_err'] = np.maximum(df.loc[rade_ok, 'pl_radeerr1'], df.loc[rade_ok, 'pl_radeerr2']) / df.loc[rade_ok, 'pl_rade']

# Quality flags
df['mass_quality'] = df['mass_frac_err'] < 0.3
df['rade_quality'] = df['rade_frac_err'] < 0.3

# Fill NaN with False
df['mass_quality'] = df['mass_quality'].fillna(False)
df['rade_quality'] = df['rade_quality'].fillna(False)

## 5. Save the dataset with quality flags

The output CSV contains all original columns plus `mass_quality` and `rade_quality`. This file is used for the analysis in the paper.

In [5]:
df.to_csv('exoplanet_data_with_quality.csv', index=False)
print("Saved exoplanet_data_with_quality.csv with", len(df), "rows")
print("Mass quality count:", df['mass_quality'].sum())
print("Radius quality count:", df['rade_quality'].sum())

Saved exoplanet_data_with_quality.csv with 6128 rows
Mass quality count: 1413
Radius quality count: 3708


## 6. Download the file (optional)

Run the cell below to download the CSV directly to your computer.

In [6]:
from google.colab import files
files.download("exoplanet_data_with_quality.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>